In [ ]:
import os, sys, warnings
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src')); warnings.filterwarnings('ignore')

# SoH predictions — vehicles with data starting 2024/2025 (smoothed)

Uses **only vehicles whose SoH data starts in 2024 or 2025** (the newer fleet). Euler has 42 such vehicles
(22 start 2024, 20 start 2025) — the **12 best-observed** are shown; Mahindra has **4** (all 2024-start).
Actual SoH is smoothed (5-pt moving average) and drawn **solid + dots through today (2026-06)**; the
**dashed line is the projection after today** (Euler = condition-aware XGB model, Mahindra = XGBoost).

In [ ]:
import numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
PAL = px.colors.qualitative.Dark24

def soh_at_age(d, age):
    A = np.concatenate([d['a_obs'], d['a_fc']]); S = np.concatenate([d['s_obs'], d['s_fc']])
    return float(np.interp(age, A, S))

def smooth(y, w=5):
    # light centred moving average to de-step the ACTUAL SoH curve (display only)
    y = np.asarray(y, dtype=float)
    if len(y) < 3:
        return y.copy()
    w = min(w, len(y)); w = w if w % 2 == 1 else w - 1; w = max(w, 3)
    pad = w // 2
    return np.convolve(np.pad(y, pad, mode='edge'), np.ones(w) / w, mode='valid')


def plot_grid(ds, title):
    # per-vehicle subplots on a CALENDAR x-axis (date), with today + warranty lines (consistent w/ the overlay)
    n = len(ds); ncol = min(3, n); nrow = int(np.ceil(n / ncol))
    fig = make_subplots(rows=nrow, cols=ncol, subplot_titles=[d['label'] for d in ds],
                        vertical_spacing=0.14, horizontal_spacing=0.06)
    for i, d in enumerate(ds):
        r, c = i // ncol + 1, i % ncol + 1
        col = 'crimson' if soh_at_age(d, d['wyr']) < 78 else 'seagreen'
        reg = pd.Timestamp(d['reg'])
        cal_o = [reg + pd.Timedelta(days=float(a) * 365.25) for a in d['a_obs']]
        cal_f = [reg + pd.Timedelta(days=float(a) * 365.25) for a in d['a_fc']]
        fig.add_scatter(x=[reg, cal_o[0]], y=[100, d['s_obs'][0]], mode='lines', row=r, col=c,
                        line=dict(color='royalblue', width=1, dash='dot'), showlegend=False, hoverinfo='skip')
        fig.add_scatter(x=cal_o, y=d['s_obs'], mode='lines+markers', row=r, col=c, name='actual (through today)',
                        line=dict(color='royalblue', width=1.5), marker=dict(size=4), legendgroup='a', showlegend=(i == 0))
        fig.add_scatter(x=cal_f, y=d['s_fc'], mode='lines', row=r, col=c, name='projection',
                        line=dict(color=col, width=2.4, dash='dash'), legendgroup='f', showlegend=(i == 0))
        fig.add_hline(y=80, line=dict(color='red', width=1, dash='dot'), row=r, col=c)
        wdate = reg + pd.DateOffset(years=int(d['wyr']))
        fig.add_scatter(x=[wdate, wdate], y=[58, 101], mode='lines', row=r, col=c, line=dict(color='green', width=1, dash='dashdot'), showlegend=False, hoverinfo='skip')
        fig.add_scatter(x=[TODAY, TODAY], y=[58, 101], mode='lines', row=r, col=c, line=dict(color='black', width=1, dash='dashdot'), showlegend=False, hoverinfo='skip')
        fig.update_xaxes(title_text='calendar', type='date', row=r, col=c, title_font_size=9)
        fig.update_yaxes(range=[58, 101], row=r, col=c)
    fig.update_layout(height=300 * nrow, width=1150, title_text=title, template='plotly_white',
                      legend=dict(orientation='h', y=1.05, x=0))
    fig.show()

def plot_overlay(ds, title, soh_label, continuous=False, show_today=True, show_warranty=True, loan_years=None, regno=None, label_rul=False, show_marker=True):
    # calendar x-axis on the BOTTOM, age on the TOP; actual solid+dots through today, dashed projection after;
    # vertical line marks today's date. (Age top axis is approximate -> anchored to the cohort median registration.)
    fig = go.Figure(); nr = ng = 0
    ref = pd.Timestamp(int(np.median([pd.Timestamp(d['reg']).value for d in ds if pd.notna(d.get('reg'))])))
    def cal(d, ages):
        return [pd.Timestamp(d['reg']) + pd.Timedelta(days=float(a) * 365.25) for a in ages]
    xmin = min(cal(d, [d['a_obs'][0]])[0] for d in ds)
    xmax = max(cal(d, [max(list(d['a_fc']) + [d['a_obs'][-1]])])[0] for d in ds)
    ymin = 58
    for i, d in enumerate(ds):
        warr = soh_at_age(d, d['wyr']); risk = warr < 78; nr += risk; ng += not risk
        tail = d['label'].split()[0]; col = PAL[i % len(PAL)]
        if label_rul:
            af = np.asarray(d['a_fc'], float); sf = np.asarray(d['s_fc'], float); now_age = (TODAY - pd.Timestamp(d['reg'])).days / 365.25
            if len(sf) and sf.min() <= 80:
                jx = int(np.argmax(sf <= 80)); age80 = np.interp(80, [sf[jx], sf[jx - 1]], [af[jx], af[jx - 1]]) if jx > 0 else af[0]
                nm = f"{(regno or {}).get(d['vin'], tail)}  ·  {max(0.0, age80 - now_age):.1f} yr to 80%"
            else:
                nm = f"{(regno or {}).get(d['vin'], tail)}  ·  >{(af[-1] - now_age) if len(af) else 0:.0f} yr to 80%"
        else:
            nm = tail + f" - {warr:.0f}% @ {d['wyr']}yr" + (' v' if warr >= 80 else '')
        fig.add_scatter(x=[pd.Timestamp(d['reg']), cal(d, [d['a_obs'][0]])[0]], y=[100, d['s_obs'][0]], mode='lines', opacity=0.45,
                        line=dict(color=col, width=1, dash='dot'), legendgroup=tail, showlegend=False, hoverinfo='skip')
        fig.add_scatter(x=cal(d, d['a_obs']), y=list(d['s_obs']), mode=('lines' if continuous else 'lines+markers'), opacity=0.95, legendgroup=tail,
                        line=dict(color=col, width=(2.4 if continuous else 1.7), shape=('spline' if continuous else 'linear')), marker=dict(size=5, color=col),
                        name=nm, showlegend=True,
                        hovertemplate=tail + ' actual %{y:.1f}%  %{x|%b %Y}<extra></extra>')
        fig.add_scatter(x=cal(d, d['a_fc']), y=list(d['s_fc']), mode='lines', opacity=0.95, legendgroup=tail, showlegend=False,
                        line=dict(color=col, width=2, dash='dash', shape=('spline' if continuous else 'linear')), hovertemplate=tail + ' projection %{y:.1f}%  %{x|%b %Y}<extra></extra>')
        if show_marker:
            wdate = pd.Timestamp(d['reg']) + pd.DateOffset(years=int(d['wyr']))
            fig.add_scatter(x=[wdate], y=[warr], mode='markers', legendgroup=tail, showlegend=False,
                            marker=dict(symbol='triangle-down', size=10, color=('crimson' if risk else 'seagreen'), line=dict(color='black', width=0.6)),
                            hovertemplate=tail + f" @{d['wyr']}yr warranty " + '%{y:.1f}%<extra></extra>')
    ymin = 58
    fig.add_hline(y=80, line=dict(color='red', width=1.3, dash='dot'), annotation_text='80% EoFL', annotation_position='bottom left')
    if show_warranty:
        for wy in sorted(set(d['wyr'] for d in ds)):
            wdate = pd.Timestamp(int(np.median([pd.Timestamp(d['reg']).value for d in ds if d['wyr'] == wy]))) + pd.DateOffset(years=int(wy))
            fig.add_scatter(x=[wdate, wdate], y=[ymin, 101], mode='lines', line=dict(color='dimgray', width=1.3, dash='dashdot'), showlegend=False,
                            hovertemplate=f'{wy}-yr warranty ~' + pd.Timestamp(wdate).strftime('%b %Y') + '<extra></extra>')
            fig.add_annotation(x=wdate, y=ymin + 1.5, text=f'{wy}-yr warranty', showarrow=False, textangle=-90, xanchor='right', yanchor='bottom', font=dict(color='dimgray', size=10))
    if loan_years:
        ldate = pd.Timestamp(int(np.median([pd.Timestamp(d['reg']).value for d in ds]))) + pd.DateOffset(years=int(loan_years))
        fig.add_scatter(x=[ldate, ldate], y=[ymin, 101], mode='lines', line=dict(color='#6a3d9a', width=1.6, dash='dashdot'), showlegend=False,
                        hovertemplate='loan tenure ends ~' + pd.Timestamp(ldate).strftime('%b %Y') + '<extra></extra>')
        fig.add_annotation(x=ldate, y=ymin + 1.5, text=f'{loan_years}-yr loan tenure ends ' + pd.Timestamp(ldate).strftime('%b %Y'), showarrow=False, textangle=-90, xanchor='right', yanchor='bottom', font=dict(color='#6a3d9a', size=10))
    if show_today:
        fig.add_scatter(x=[TODAY, TODAY], y=[ymin, 101], mode='lines', line=dict(color='black', width=1.4, dash='dashdot'), showlegend=False, hoverinfo='skip')
        fig.add_annotation(x=TODAY, y=101, text=TODAY.strftime('%d %b %Y'), showarrow=False, yanchor='bottom', xanchor='left', font=dict(color='black', size=11))
    yrt = pd.date_range(pd.Timestamp(xmin).to_period('Y').to_timestamp(), pd.Timestamp(xmax) + pd.Timedelta(days=370), freq='YS')
    age_txt = [f"{(t - ref).days / 365.25:.0f}" for t in yrt]
    fig.add_scatter(x=[xmin], y=[ymin], xaxis='x2', mode='markers', marker=dict(opacity=0), showlegend=False, hoverinfo='skip')
    fig.update_layout(height=640, width=1200, template='plotly_white', margin=dict(t=72),
                      title=dict(text=(title if label_rul else f"{title}  -  survives {ng} / at-risk {nr}"), y=0.97),
                      xaxis=dict(title='calendar (year-month)', type='date', range=[xmin, xmax]),
                      xaxis2=dict(overlaying='x', side='top', type='date', range=[xmin, xmax], tickmode='array',
                                  tickvals=list(yrt), ticktext=age_txt, title=dict(text='age since registration (yr)', standoff=6),
                                  tickfont=dict(size=9), title_font=dict(size=10)),
                      yaxis=dict(title='SoH (%)', range=[ymin, 101]),
                      legend=dict(title=('vehicle · RUL to 80% SoH' if label_rul else 'vehicle - SoH @ warranty'), x=1.01, y=1, font=dict(size=10)))
    fig.show()


TODAY = pd.Timestamp('2026-06-19')

def split_at(d, today_age):
    # show predicted points up to TODAY as solid+dots (like data); keep only the post-today tail as the dashed projection
    af = np.asarray(d['a_fc'], float); sf = np.asarray(d['s_fc'], float)
    pre = af <= today_age; post = ~pre
    if pre.any():
        d['a_obs'] = np.concatenate([np.asarray(d['a_obs'], float), af[pre]])
        d['s_obs'] = np.concatenate([np.asarray(d['s_obs'], float), sf[pre]])
    seed_a = af[pre][-1:] if (pre.any() and post.any()) else np.array([])
    seed_s = sf[pre][-1:] if (pre.any() and post.any()) else np.array([])
    d['a_fc'] = np.concatenate([seed_a, af[post]]); d['s_fc'] = np.concatenate([seed_s, sf[post]])

print('helpers ready')

## Euler — data starting 2024/2025 (12 best-observed)

In [ ]:
import euler_features, euler_model as em
if not Path('data/euler/features/feature_table.parquet').exists():
    euler_features.main()
meu = pd.read_parquet('data/euler/features/feature_table.parquet').sort_values(['vin', 'month'])
ereg = pd.read_csv('data/euler/Euler_Regd_Details.csv')
ereg['reg'] = pd.to_datetime(ereg['regd_date'], format='%d/%m/%y', errors='coerce')
EREG = dict(zip(ereg['vin'], ereg['reg']))
emodel = em.train(em.build_transitions(meu))
_er = pd.read_csv('data/euler/Euler_Regd_Details.csv').drop_duplicates('vin'); _m1 = dict(zip(_er['vin'], _er['regd_number']))
_sold = pd.read_csv('data/Repo - Pricing - Sold.csv'); _sold.columns = [c.strip() for c in _sold.columns]; _m2 = dict(zip(_sold['VIN'], _sold['Vehicle Reg No']))
REGNO = {v: (_m1.get(v) or _m2.get(v) or v[-6:]) for v in meu['vin'].unique()}
_ef = meu.groupby('vin')['month'].min(); _en = meu.groupby('vin').size()
ecand = [v for v in meu['vin'].unique() if _ef[v].year in (2024, 2025)]
ejun = sorted(ecand, key=lambda v: _en[v], reverse=True)[:12]   # 12 best-observed -> accurate forecasts

def eu_d(vin):
    g = meu[meu['vin'] == vin].sort_values('month')
    reg_t = EREG.get(vin); reg_t = pd.Timestamp(reg_t) if pd.notna(reg_t) else g['month'].iloc[0]
    last_age = float(g['age_months'].iloc[-1]); H = max(int(round(72 - last_age)), 1)
    fc = np.array(em.free_run(g, emodel, H))
    a_obs = ((g['month'] - reg_t).dt.days / 365.25).to_numpy()
    a_fc = np.array([((g['month'].iloc[-1] + pd.DateOffset(months=k)) - reg_t).days / 365.25 for k in range(1, H + 1)])
    d = dict(a_obs=a_obs, s_obs=smooth(g['soh'].to_numpy()), a_fc=a_fc, s_fc=fc, wyr=5, reg=reg_t, vin=vin)
    d['label'] = f"{vin[-6:]}  now {d['s_obs'][-1]:.0f}% -> 5yr ~{soh_at_age(d,5):.0f}%"
    return d
euler_ds = [eu_d(v) for v in ejun]
for d in euler_ds:
    split_at(d, (TODAY - pd.Timestamp(d['reg'])).days / 365.25)
print(f'{len(ecand)} Euler vehicles with data starting 2024/2025; showing {len(euler_ds)} best-observed:', [d['label'].split()[0] for d in euler_ds])

In [ ]:
if euler_ds:
    plot_grid(euler_ds, 'Euler (data starting 2024/2025) — smoothed BMS SoH; solid=through today, dashed=projection')
else:
    print('no Euler data-start 2024/2025 vehicles')

In [ ]:
if euler_ds:
    plot_overlay(euler_ds, 'Euler (data starting 2024/2025) — overlay (solid=through today, dashed=projection)', 'BMS SoH')
else:
    print('no Euler data-start 2024/2025 vehicles')

In [ ]:
# === Plot ONLY the selected Euler vehicles (calendar bottom, age top, dated today line) ===
SEL = ['217125', '217170', '217054']
sub = [d for d in euler_ds if d['vin'][-6:] in SEL]
plot_overlay(sub, 'Euler SoH Prediction', 'BMS SoH', show_today=True, show_warranty=False, loan_years=3, regno=REGNO, label_rul=True, show_marker=False)

In [ ]:
# === Plot ONLY these selected Euler vehicles (calendar bottom, age top, dated today line) ===
SEL2 = ['217170', '217054', '217135']
sub2 = [d for d in euler_ds if d['vin'][-6:] in SEL2]
plot_overlay(sub2, 'Euler SoH Prediction', 'BMS SoH', show_today=True, show_warranty=False, loan_years=3, regno=REGNO, label_rul=True, show_marker=False)

In [ ]:
# === Continuous smooth curves (spline) for the selected vehicles ===
plot_overlay(sub2, 'Euler SoH Prediction', 'BMS SoH', continuous=True, show_today=True, show_warranty=False, loan_years=3, regno=REGNO, label_rul=True, show_marker=False)

In [ ]:
# === Euler SoH Prediction - blue vehicle removed, anonymised legend ===
subv = [d for d in euler_ds if d['vin'][-6:] in ['217054', '217135']]
VNAMES = {d['vin']: f'Vehicle {i + 1:03d}' for i, d in enumerate(subv)}
plot_overlay(subv, 'Euler SoH Prediction', 'BMS SoH', show_today=True, show_warranty=False, loan_years=3, regno=VNAMES, label_rul=True, show_marker=False)

## Mahindra — data starting 2024 (coulomb SoH)

In [ ]:
import model as mhm, xgboost as xgb, config
mh = pd.read_parquet('data/mahindra/features/feature_table.parquet').sort_values(['vin', 'month'])
trh = mhm.build_transitions(mh)
mhx = xgb.XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=4, subsample=0.8,
                       colsample_bytree=0.8, n_jobs=8, verbosity=0).fit(
    trh[mhm.FEATS].to_numpy(), trh['loss'].to_numpy(), sample_weight=trh['w'].to_numpy())
mreg = pd.read_csv('Mh_Regd_Date.csv'); mreg['reg'] = pd.to_datetime(mreg['vehicle_registration_date'], errors='coerce')
MREG = dict(zip(mreg['vin'], mreg['reg']))
vmod = dict(pd.read_csv('data/manifests/mahindra_vin_model.csv').values)
mfirst = mh.groupby('vin')['month'].min()
mjun = [v for v in mfirst.index if mfirst[v].year in (2024, 2025) and (mh['vin'] == v).sum() >= 4]

def mh_d(vin):
    g = mh[mh['vin'] == vin].sort_values('month'); last = g.iloc[-1]
    reg_t = MREG.get(vin)
    reg_t = reg_t if pd.notna(reg_t) else (g['month'].iloc[0] - pd.Timedelta(days=float(g['age_months'].iloc[0]) * 30.4375))
    last_age = float(last['age_months']); H = max(int(round(60 - last_age)), 1)
    s_obs = smooth(g['soh'].to_numpy())
    stress = g.iloc[-6:][mhm.STRESS].median().to_dict(); st = {s: float(last[s]) for s in mhm.STATE}; st['soh'] = float(s_obs[-1]); soh = float(s_obs[-1]); fc = []
    for _ in range(H):
        isa, dfc = mhm._curv(st['age_months'], st['soh'])
        x = pd.DataFrame([{**{s: st[s] for s in mhm.STATE}, **stress, 'inv_sqrt_age': isa, 'soh_deficit': dfc}])[mhm.FEATS].to_numpy()
        soh = soh - max(mhx.predict(x)[0], 0)
        st.update(soh=soh, age_months=st['age_months'] + 1, cum_ah=st['cum_ah'] + stress.get('ah_throughput', 0)); fc.append(soh)
    a_obs = ((g['month'] - reg_t).dt.days / 365.25).to_numpy()
    a_fc = np.array([((g['month'].iloc[-1] + pd.DateOffset(months=k)) - reg_t).days / 365.25 for k in range(1, H + 1)])
    wyr = config.warranty_for('mahindra', vmod.get(vin, ''))[0]
    d = dict(a_obs=a_obs, s_obs=s_obs, a_fc=a_fc, s_fc=np.array(fc), wyr=wyr, vin=vin, reg=reg_t)
    d['label'] = f"{vin[-6:]}  now {d['s_obs'][-1]:.0f}% -> {wyr}yr ~{soh_at_age(d,wyr):.0f}%"
    return d
mh_ds = [mh_d(v) for v in mjun]
for d in mh_ds:
    split_at(d, (TODAY - pd.Timestamp(d['reg'])).days / 365.25)
print(f'{len(mh_ds)} Mahindra vehicles with data starting 2024/2025:', [d['label'].split()[0] for d in mh_ds])

(plot_grid(mh_ds, 'Mahindra (data starting 2024/2025) — smoothed coulomb SoH; solid=through today, dashed=projection') if mh_ds else print('no Mahindra data-start 2024/2025 vehicles'))

mh_3 = [d for d in mh_ds if d['wyr'] == 3]   # Treo
mh_5 = [d for d in mh_ds if d['wyr'] == 5]   # Zor Grand
print(f"3-yr warranty (Treo): {len(mh_3)} | 5-yr warranty (Zor Grand): {len(mh_5)}")
if mh_3:
    plot_overlay(mh_3, 'Mahindra Treo — 3-yr warranty (data starting 2024/2025)', 'coulomb SoH')
if mh_5:
    plot_overlay(mh_5, 'Mahindra Zor Grand — 5-yr warranty (data starting 2024/2025)', 'coulomb SoH')